# 08 · 序列模型：RNN / LSTM / GRU

> 序列数据 = 有顺序、前后相关的数据：文本、语音、时间序列、DNA……
> 循环神经网络（RNN）通过"隐状态"传递历史信息。

## 8.1 为什么需要循环结构

普通全连接/CNN 假设样本独立、输入定长。序列数据的挑战：
- **变长**：句子长度不同
- **顺序敏感**："猫追狗" vs "狗追猫"
- **长程依赖**：句首的主语影响句尾的动词

RNN 的核心：**同一个模块重复使用，隐状态 $h_t$ 携带历史**：

$$
h_t = \tanh(W_{ih}x_t + b_{ih} + W_{hh}h_{t-1} + b_{hh})
$$

## 8.2 输入输出形状

```
(batch, seq_len, input_size)  ->  RNN  ->  output: (batch, seq_len, hidden_size)
                                          h_n   : (num_layers, batch, hidden_size)
```

关键参数 `batch_first=True`（默认 False，即 `(seq, batch, feat)`）。**强烈建议设为 True**。

In [ ]:
import torch
import torch.nn as nn

batch, seq_len, input_size, hidden = 4, 5, 3, 8

# batch_first=False（默认）
rnn = nn.RNN(input_size, hidden, batch_first=False)
x = torch.randn(seq_len, batch, input_size)
out, h_n = rnn(x)
print("默认 输入:", x.shape, "输出:", out.shape, "h_n:", h_n.shape)

# batch_first=True（推荐）
rnn = nn.RNN(input_size, hidden, num_layers=2, batch_first=True, bidirectional=True)
x = torch.randn(batch, seq_len, input_size)
out, h_n = rnn(x)
print("\nbatch_first 输入:", x.shape)
print("输出 out:", out.shape, "  # 每个时间步的隐状态（双向则 2*hidden）")
print("末态 h_n:", h_n.shape, "  # (num_layers*num_directions, batch, hidden)")

## 8.3 三种循环单元

| 单元 | 门 | 特点 |
|---|---|---|
| `nn.RNN` | 无 | 最简单，长序列易梯度消失 |
| `nn.LSTM` | 输入门、遗忘门、输出门 + 细胞状态 | 能记长程依赖，最常用 |
| `nn.GRU` | 重置门、更新门 | 比 LSTM 少一个门，更快，效果接近 |

它们都是 `nn.Module`，用法几乎一样。

In [ ]:
for cls in [nn.RNN, nn.LSTM, nn.GRU]:
    rnn = cls(10, 20, num_layers=2, batch_first=True)
    n = sum(p.numel() for p in rnn.parameters())
    print(f"{cls.__name__:5s} 参数量: {n}")

# LSTM 返回 3 个值
lstm = nn.LSTM(10, 20, batch_first=True)
x = torch.randn(4, 7, 10)
out, (h_n, c_n) = lstm(x)
print("\nLSTM 输出:", out.shape, "h_n:", h_n.shape, "c_n:", c_n.shape)

## 8.4 手动实现一个 RNN 单元（理解原理）

$$h_t = \tanh(x_t W_{ih}^T + h_{t-1} W_{hh}^T + b)$$

In [ ]:
class ManualRNN(nn.Module):
    def __init__(self, input_size, hidden):
        super().__init__()
        self.hidden = hidden
        self.W_ih = nn.Linear(input_size, hidden)   # 输入变换
        self.W_hh = nn.Linear(hidden, hidden)       # 隐状态变换

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        B, T, _ = x.shape
        h = torch.zeros(B, self.hidden, device=x.device)
        outs = []
        for t in range(T):
            h = torch.tanh(self.W_ih(x[:, t]) + self.W_hh(h))
            outs.append(h)
        return torch.stack(outs, dim=1), h

m = ManualRNN(3, 8)
x = torch.randn(4, 5, 3)
out, h = m(x)
print("手动 RNN 输出:", out.shape, " h:", h.shape)

# 与 nn.RNN 对比形状（值不同，因为权重不同）
ref = nn.RNN(3, 8, batch_first=True)
print("nn.RNN 输出:", ref(x)[0].shape)

## 8.5 示例一：正弦波预测（序列回归）

任务：用前 20 个点预测第 21 个点。

**关键技术：构造滑动窗口数据集**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)

def make_sine_data(n_points=1000, window=20, noise=0.0, seed=0):
    rng = np.random.RandomState(seed)
    t = np.linspace(0, 40 * np.pi, n_points)
    y = np.sin(t) + noise * rng.randn(n_points)
    X, Y = [], []
    for i in range(len(y) - window):
        X.append(y[i:i + window])
        Y.append(y[i + window])
    X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1)  # (N, window, 1)
    Y = torch.tensor(np.array(Y), dtype=torch.float32).unsqueeze(-1)  # (N, 1)
    return X, Y

X, Y = make_sine_data(600, window=20, noise=0.05)
print("X:", X.shape, " Y:", Y.shape)

n_train = 500
X_tr, Y_tr = X[:n_train], Y[:n_train]
X_te, Y_te = X[n_train:], Y[n_train:]
print("训练:", X_tr.shape, " 测试:", X_te.shape)

In [ ]:
plt.figure(figsize=(12, 3))
plt.plot(Y[:200].squeeze(), label="target")
plt.title("正弦序列（目标值）")
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
class SineRNN(nn.Module):
    def __init__(self, hidden=32, num_layers=1, cell="LSTM"):
        super().__init__()
        cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[cell]
        self.rnn = cls(1, hidden, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)          # (B, T, H)
        last = out[:, -1]             # 取最后一个时间步
        return self.fc(last)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def train_sine(cell, hidden=32, epochs=30, lr=1e-2):
    model = SineRNN(hidden=hidden, cell=cell).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()
    Xt, Yt = X_tr.to(device), Y_tr.to(device)
    Xe, Ye = X_te.to(device), Y_te.to(device)
    for e in range(epochs):
        model.train()
        opt.zero_grad()
        loss = crit(model(Xt), Yt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    model.eval()
    with torch.no_grad():
        test_loss = crit(model(Xe), Ye).item()
    return model, loss.item(), test_loss

results = {}
for cell in ["RNN", "LSTM", "GRU"]:
    model, tl, vl = train_sine(cell)
    results[cell] = (model, tl, vl)
    print(f"{cell:5s} | 训练 MSE {tl:.5f} | 测试 MSE {vl:.5f}")

In [ ]:
# 可视化预测
model, _, _ = results["LSTM"]
model.eval()
with torch.no_grad():
    pred = model(X_te.to(device)).cpu().squeeze()

plt.figure(figsize=(12, 4))
plt.plot(Y_te.squeeze(), label="真实", alpha=0.7)
plt.plot(pred, label="预测", linestyle="--")
plt.title("LSTM 正弦波预测")
plt.xlabel("时间步"); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 8.6 示例二：字符级文本分类（名字→语言）

经典的"多对一"任务：读一串字符，输出一个类别。
这里用自造的随机字符串演示流程（避免依赖外部数据集）。

In [ ]:
import string

def make_name_data(n=600, seed=0):
    rng = np.random.RandomState(seed)
    langs = ["alpha", "beta", "gamma"]
    X_str, y = [], []
    for _ in range(n):
        c = rng.randint(0, 3)
        L = rng.randint(4, 10)
        if c == 0:
            s = "".join(rng.choice(list(string.ascii_lowercase[:13]), L))    # a-m
        elif c == 1:
            s = "".join(rng.choice(list(string.ascii_lowercase[13:]), L))    # n-z
        else:
            s = "".join(rng.choice(list("aeiou"), L))
        X_str.append(s); y.append(c)
    return X_str, torch.tensor(y)

X_str, y = make_name_data()
vocab = list(string.ascii_lowercase)
char2idx = {c: i for i, c in enumerate(vocab)}
V = len(vocab)
print("词表大小:", V, " 样本数:", len(X_str))
print("样例:", X_str[:5], y[:5].tolist())

In [ ]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

class NameDataset(Dataset):
    def __init__(self, X_str, y, char2idx):
        self.seqs = [torch.tensor([char2idx[c] for c in s]) for s in X_str]
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.seqs[i], self.y[i]

def collate(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = pad_sequence(seqs, batch_first=True)      # (B, T_max)
    return padded, lengths, torch.stack(labels)

ds = NameDataset(X_str, y, char2idx)
n = int(0.8 * len(ds))
tr_ds, te_ds = torch.utils.data.random_split(ds, [n, len(ds) - n],
                                             generator=torch.Generator().manual_seed(0))
tr_loader = DataLoader(tr_ds, batch_size=32, shuffle=True, collate_fn=collate)
te_loader = DataLoader(te_ds, batch_size=64, collate_fn=collate)
print("train batches:", len(tr_loader), " test batches:", len(te_loader))

In [ ]:
class NameClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=16, hidden=32, num_classes=3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        # x: (B, T) 整数索引
        e = self.emb(x)                 # (B, T, emb)
        out, h = self.gru(e)            # out: (B,T,2H)
        # 用最后时间步（这里未做 padding mask，简单演示）
        last = out[:, -1]               # (B, 2H)
        return self.fc(last)

model = NameClassifier(V).to(device)
print(model)
print("输出:", model(torch.randint(1, V, (4, 7)).to(device)).shape)

In [ ]:
crit = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)

for epoch in range(1, 11):
    model.train()
    tl, corr, n = 0, 0, 0
    for x, lengths, y in tr_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = crit(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        tl += loss.item() * len(y); corr += (logits.argmax(1) == y).sum().item(); n += len(y)

    model.eval()
    tc, tn = 0, 0
    with torch.no_grad():
        for x, lengths, y in te_loader:
            x, y = x.to(device), y.to(device)
            tc += (model(x).argmax(1) == y).sum().item(); tn += len(y)

    if epoch % 2 == 0:
        print(f"epoch {epoch:2d} | train_loss {tl/n:.4f} acc {corr/n:.3f} | test acc {tc/tn:.3f}")

## 8.7 梯度裁剪：RNN 的救命稻草

RNN 反向传播沿时间展开（BPTT），梯度容易**爆炸**。

$$
\|\nabla\| > \text{threshold} \Rightarrow \nabla \leftarrow \nabla \cdot \frac{\text{threshold}}{\|\nabla\|}
$$

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

In [ ]:
# 演示：不裁剪 vs 裁剪
torch.manual_seed(0)
model_a = nn.RNN(1, 32, batch_first=True)
x = torch.randn(4, 50, 1) * 5
out, _ = model_a(x)
out.sum().backward()
norms = [p.grad.norm().item() for p in model_a.parameters() if p.grad is not None]
print("未裁剪的梯度范数:", [round(n, 3) for n in norms])

model_b = nn.RNN(1, 32, batch_first=True)
out, _ = model_b(x)
out.sum().backward()
torch.nn.utils.clip_grad_norm_(model_b.parameters(), 1.0)
norms_b = [p.grad.norm().item() for p in model_b.parameters() if p.grad is not None]
print("裁剪后的梯度范数:", [round(n, 3) for n in norms_b])

## 8.8 处理 padding：pack_padded_sequence

变长序列补 0 后，padding 位置不应参与计算。用 `pack_padded_sequence` 提高效率与精度。

```python
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
out, h = rnn(packed)
out, out_lengths = pad_packed_sequence(out, batch_first=True)
```

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

x, lengths, y = next(iter(tr_loader))
emb = nn.Embedding(V, 16, padding_idx=0)
e = emb(x)                       # (B, T, 16)
gru = nn.GRU(16, 32, batch_first=True)

# 普通做法
out1, h1 = gru(e)
# packed 做法
packed = pack_padded_sequence(e, lengths, batch_first=True, enforce_sorted=False)
out2, h2 = gru(packed)
out2, lens2 = pad_packed_sequence(out2, batch_first=True)

print("普通 out:", out1.shape, " packed 还原 out:", out2.shape)
print("h_n 形状相同:", h1.shape, h2.shape)
print("有效长度示例:", lengths[:5].tolist())

### 用 attention pooling 或取有效最后一步

变长序列做分类时，正确做法是取每个样本**真实最后一步**的隐状态：
```python
last = out[torch.arange(B), lengths - 1]
```

In [ ]:
B, T, H = out1.shape
idx = torch.arange(B)
last_correct = out1[idx, lengths - 1]         # 每个样本的真实最后一步
print("正确取最后一步:", last_correct.shape)
print("（对比 out1[:, -1] 会取到 padding 部分，对变长序列是错的）")

## 8.9 双向与多层

```python
nn.LSTM(input, hidden, num_layers=2, bidirectional=True, dropout=0.3, batch_first=True)
```
- **多层**：堆叠加深，`num_layers` 层之间有 dropout（需 >1 才生效）。
- **双向**：正序 + 倒序各跑一遍，输出拼接，适合"能看到整句"的任务（分类/标注），
  不适合自回归生成（未来不可见）。

## 8.10 练习

**练习 1**：把正弦预测的 `input window` 从 20 改成 50，看测试 MSE。
用普通 RNN、LSTM、GRU 各跑一次，比较谁最能记住长依赖。

**练习 2**：给 `SineRNN` 加 `num_layers=2, dropout=0.2`，比较效果。

**练习 3**：名字分类中，把 `bidirectional=True` 改成 False，准确率变化。

**练习 4**：把 `out[:, -1]` 改成 `out[torch.arange(B), lengths-1]`，看准确率是否提升。

**练习 5**：写一个"多对多"模型：输入序列，输出等长序列（每个时间步都预测），
用于序列标注（如每个字符预测是否为元音）。

In [ ]:
# 练习 1：更长依赖
for window in [20, 50]:
    X2, Y2 = make_sine_data(1200, window=window, noise=0.05)
    global X_tr, Y_tr, X_te, Y_te
    X_tr, Y_tr, X_te, Y_te = X2[:1000], Y2[:1000], X2[1000:], Y2[1000:]
    print(f"\nwindow={window}")
    for cell in ["RNN", "LSTM", "GRU"]:
        _, tl, vl = train_sine(cell, epochs=30)
        print(f"  {cell:5s} test MSE {vl:.5f}")

In [ ]:
# 练习 5：多对多（逐时间步二分类：是否为元音）
class Many2Many(nn.Module):
    def __init__(self, vocab, emb=16, hidden=32):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb)
        self.gru = nn.GRU(emb, hidden, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden * 2, 2)
    def forward(self, x):
        return self.fc(self.gru(self.emb(x))[0])    # (B, T, 2)

m = Many2Many(V)
x = torch.randint(1, V, (4, 7))
print("练习5 多对多输出:", m(x).shape)

## 8.11 小结

- 形状约定：`batch_first=True` → 输入 `(B, T, D)`，输出 `(B, T, H)`。
- RNN 简单但易梯度消失；**LSTM/GRU 是实际首选**。
- 序列任务要**梯度裁剪**（`clip_grad_norm_`）。
- 变长序列用 `pack_padded_sequence`，分类取 `out[arange(B), lengths-1]`。
- 双向 + 多层可提升表达力；自回归任务不能用双向。

下一节：`09_注意力机制与Transformer.ipynb`